# See LangChain chunking: boundaries, outputs, and tradeoffs
A self-contained Python 3.11+ teaching notebook. All sample text and setup instructions live here; no downloads of documents, vector database, or retrieval framework are needed.

**Chunking** splits source material into pieces for embedding, retrieval, or model context. A boundary can preserve a useful answer or separate it from its conditions. Smaller chunks can improve precision but lose context; larger chunks preserve context but can retrieve unrelated material.

We reuse **one support knowledge-base passage** for character, recursive, token, sentence, overlap, and semantic demonstrations. Separate Markdown, HTML, Python, and JSON fixtures are necessary to demonstrate their actual structure.

Every example prints every chunk, its size, and any metadata. Ordinary methods run locally. Semantic chunking uses real Gemini embeddings by default, with an OpenAI switch. Embeddings measure meaning; a chat LLM is not required. API credentials are needed only for that section. No fabricated semantic outputs are included.

## Strategies covered at a glance
1. **Fixed character windows:** Cut text into fixed-length character chunks, even across words or sentences.
2. **Separator-based character splitting:** Keep paragraph units together, merging them toward a target size.
3. **Recursive character splitting:** Try paragraphs, lines, words, then characters to fit a size limit.
4. **Sliding overlap:** Repeat part of each chunk in the next chunk to retain boundary context.
5. **Token-based splitting:** Split by tokenizer token counts rather than character counts.
6. **Sentence-boundary grouping:** Group whole sentences toward a target size using a punctuation-based regex.
7. **Markdown heading splitting:** Split Markdown into sections and preserve heading hierarchy as metadata.
8. **HTML heading splitting:** Split HTML by headings and retain section context as metadata.
9. **Python code-aware splitting:** Prefer class and function boundaries when recursively splitting Python code.
10. **Recursive JSON splitting:** Divide nested JSON into structured fragments that preserve parent key paths.
11. **Semantic chunking:** Use Gemini or OpenAI embeddings to split at changes in meaning between sentence windows.

## Install once
Uncomment the first line below if needed. `%pip` installs into the selected notebook kernel. Restart the kernel after installation if imports fail. No installation occurs automatically when you run all cells.


In [1]:
# %pip install "langchain-text-splitters>=0.3,<2" "langchain-core>=0.3,<2" "tiktoken>=0.9,<1" "beautifulsoup4>=4.12,<5" "lxml>=5,<7" "langchain-experimental>=0.3,<1" "langchain-google-genai>=2,<5" "langchain-openai>=0.3,<3"
# If using a fresh environment, install Jupyter separately: python -m pip install jupyterlab ipykernel
print('Install dependencies if needed, then continue.')


Install dependencies if needed, then continue.


## Shared text: a real retrieval scenario
A support assistant must answer questions such as: **“Can I get a refund after cancellation?”**, **“What if my recovery codes are lost?”**, and **“Why did my large upload fail?”**

Our synthetic article contains several sentences per topic, followed by a topic shift. Paragraphs make structure-aware splitting useful; the semantic section also removes paragraph breaks to simulate extracted or flattened text. It keeps the same wording. Print the source before looking at its chunks.


In [2]:
from langchain_text_splitters import (
    CharacterTextSplitter, RecursiveCharacterTextSplitter, TokenTextSplitter,
    MarkdownHeaderTextSplitter, HTMLHeaderTextSplitter,
    RecursiveJsonSplitter, Language,
)
from langchain_core.documents import Document
import json, re
from importlib.metadata import version

TEXT = """Customers can cancel a paid subscription from the billing page at any time. Cancellation stops the next renewal but keeps paid features active until the current billing period ends. Canceling a subscription does not automatically issue a refund. Refund requests must be submitted to billing support within fourteen days of the original purchase. Include the invoice number so the billing team can locate the payment.

Account security uses a password and an optional authenticator app. If an authenticator device is lost, a saved recovery code can restore access. Each recovery code works only once and should be stored somewhere private. If both the device and recovery codes are unavailable, contact account support for identity verification. Never send a password or a recovery code in a support ticket.

File uploads accept PDF and plain text documents up to twenty megabytes per file. A file above that limit must be compressed or divided before it can be uploaded. A failed upload does not consume the document processing quota. If a smaller file still fails, check its format and retry after the network connection is stable. Once an upload succeeds, processing can take several minutes before the document becomes searchable."""
FLAT_TEXT = ' '.join(TEXT.split())  # Same content; paragraph layout is lost.
print(TEXT)
print('\nCharacters:', len(TEXT))
print('Splitter package:', version('langchain-text-splitters'))

results = {}
def show_chunks(name, chunks, unit='characters'):
    # Normalize strings and Documents without hiding metadata or whitespace.
    docs = [c if isinstance(c, Document) else Document(page_content=c) for c in chunks]
    results[name] = docs
    print(f'\n{name}: {len(docs)} chunks | displayed length unit: {unit}')
    for i, doc in enumerate(docs, 1):
        print('\n' + '=' * 72)
        print(f'CHUNK {i} | {len(doc.page_content)} characters')
        if doc.metadata:
            print('Metadata:', doc.metadata)
        print(doc.page_content)
        print('Whitespace view:', repr(doc.page_content))
    return docs


Customers can cancel a paid subscription from the billing page at any time. Cancellation stops the next renewal but keeps paid features active until the current billing period ends. Canceling a subscription does not automatically issue a refund. Refund requests must be submitted to billing support within fourteen days of the original purchase. Include the invoice number so the billing team can locate the payment.

Account security uses a password and an optional authenticator app. If an authenticator device is lost, a saved recovery code can restore access. Each recovery code works only once and should be stored somewhere private. If both the device and recovery codes are unavailable, contact account support for identity verification. Never send a password or a recovery code in a support ticket.

File uploads accept PDF and plain text documents up to twenty megabytes per file. A file above that limit must be compressed or divided before it can be uploaded. A failed upload does not cons

## 1. Fixed character windows
Use an empty separator to split at individual characters, then merge into windows of at most 220 characters. This is a true hard-cut baseline, unlike paragraph-aware CharacterTextSplitter.

**Strength:** Predictable size, fast, no model or tokenizer.

**Weakness:** Can cut words, sentences, and answer conditions in half. Character length is not token length.

**Use cases:** A baseline or tightly bounded uniform text.

**Observe:** words may be cut across chunk boundaries. The last chunk can be shorter.


In [3]:
fixed = CharacterTextSplitter(separator='', chunk_size=220, chunk_overlap=0)
show_chunks('fixed characters', fixed.split_text(TEXT));



fixed characters: 6 chunks | displayed length unit: characters

CHUNK 1 | 220 characters
Customers can cancel a paid subscription from the billing page at any time. Cancellation stops the next renewal but keeps paid features active until the current billing period ends. Canceling a subscription does not auto
Whitespace view: 'Customers can cancel a paid subscription from the billing page at any time. Cancellation stops the next renewal but keeps paid features active until the current billing period ends. Canceling a subscription does not auto'

CHUNK 2 | 219 characters
matically issue a refund. Refund requests must be submitted to billing support within fourteen days of the original purchase. Include the invoice number so the billing team can locate the payment.

Account security uses
Whitespace view: 'matically issue a refund. Refund requests must be submitted to billing support within fourteen days of the original purchase. Include the invoice number so the billing team can locate t

## 2. Separator-based character splitting
Split on paragraph boundaries and merge paragraphs when they fit. The size target is 220 characters, smaller than our paragraphs.

**Strength:** Keeps paragraph units together.

**Weakness:** Does not recursively split an oversized paragraph: chunk_size is a target, not a hard cap here.

**Use cases:** Clean paragraph-based articles where paragraph sizes are acceptable.

**Observe:** entire paragraphs exceed 220 characters; compare them with the hard cuts above.


In [4]:
paragraphs = CharacterTextSplitter(separator='\n\n', chunk_size=220, chunk_overlap=0)
show_chunks('paragraph separators', paragraphs.split_text(TEXT));



paragraph separators: 3 chunks | displayed length unit: characters

CHUNK 1 | 416 characters
Customers can cancel a paid subscription from the billing page at any time. Cancellation stops the next renewal but keeps paid features active until the current billing period ends. Canceling a subscription does not automatically issue a refund. Refund requests must be submitted to billing support within fourteen days of the original purchase. Include the invoice number so the billing team can locate the payment.
Whitespace view: 'Customers can cancel a paid subscription from the billing page at any time. Cancellation stops the next renewal but keeps paid features active until the current billing period ends. Canceling a subscription does not automatically issue a refund. Refund requests must be submitted to billing support within fourteen days of the original purchase. Include the invoice number so the billing team can locate the payment.'

CHUNK 2 | 388 characters
Account security uses a pas

Created a chunk of size 416, which is longer than the specified 220
Created a chunk of size 388, which is longer than the specified 220


## 3. Recursive character splitting
Try paragraph boundaries, then line breaks, spaces, and finally individual characters. Smaller separators are used only when needed.

**Strength:** A strong general baseline that balances readable boundaries and bounded size.

**Weakness:** Can still divide a sentence or topic. It does not understand meaning.

**Use cases:** General documents and a first RAG prototype.

**Observe:** chunks stay at or below 220 characters with this fallback list but often end at word boundaries.


In [5]:
recursive = RecursiveCharacterTextSplitter(
    chunk_size=220, chunk_overlap=0, separators=['\n\n', '\n', ' ', ''])
show_chunks('recursive characters', recursive.split_text(TEXT));



recursive characters: 6 chunks | displayed length unit: characters

CHUNK 1 | 215 characters
Customers can cancel a paid subscription from the billing page at any time. Cancellation stops the next renewal but keeps paid features active until the current billing period ends. Canceling a subscription does not
Whitespace view: 'Customers can cancel a paid subscription from the billing page at any time. Cancellation stops the next renewal but keeps paid features active until the current billing period ends. Canceling a subscription does not'

CHUNK 2 | 200 characters
automatically issue a refund. Refund requests must be submitted to billing support within fourteen days of the original purchase. Include the invoice number so the billing team can locate the payment.
Whitespace view: 'automatically issue a refund. Refund requests must be submitted to billing support within fourteen days of the original purchase. Include the invoice number so the billing team can locate the payment.'

CHUNK 3

## 4. Sliding overlap: repeat context across boundaries
Overlap is a setting, not a separate splitter class. Use identical text, splitter, and size to isolate its effect. A 50-character target repeats context in neighboring windows.

**Strength:** Can preserve clues needed near a boundary.

**Weakness:** Duplicates stored text, embeddings, and retrieved content. Recursive overlap may be smaller than the target because it respects split units.

**Use cases:** Retrieval where answers often straddle boundaries.

**Observe:** the suffix of one chunk appears at the start of the next. This hard-cut example makes the overlap exact.


In [6]:
overlapping = CharacterTextSplitter(separator='', chunk_size=220, chunk_overlap=50)
show_chunks('fixed with overlap', overlapping.split_text(TEXT))
a, b = results['fixed with overlap'][:2]
print('Exact 50-character shared boundary:', a.page_content[-50:] == b.page_content[:50])



fixed with overlap: 7 chunks | displayed length unit: characters

CHUNK 1 | 220 characters
Customers can cancel a paid subscription from the billing page at any time. Cancellation stops the next renewal but keeps paid features active until the current billing period ends. Canceling a subscription does not auto
Whitespace view: 'Customers can cancel a paid subscription from the billing page at any time. Cancellation stops the next renewal but keeps paid features active until the current billing period ends. Canceling a subscription does not auto'

CHUNK 2 | 220 characters
eriod ends. Canceling a subscription does not automatically issue a refund. Refund requests must be submitted to billing support within fourteen days of the original purchase. Include the invoice number so the billing te
Whitespace view: 'eriod ends. Canceling a subscription does not automatically issue a refund. Refund requests must be submitted to billing support within fourteen days of the original purchase. Includ

## 5. Token-based splitting
Count cl100k_base tokens rather than characters. Tokenization is model-specific; this tokenizer demonstrates the idea and is not the Gemini tokenizer.

**Strength:** Useful for context budgets and embedding limits.

**Weakness:** Can cut natural language boundaries. Direct token splitting can damage some Unicode characters when token boundaries divide their encoding.

**Use cases:** Token-constrained pipelines; use the actual target model tokenizer for budgeting.

**Observe:** each chunk has at most 55 tokens under this encoding; character lengths vary.


In [ ]:
import tiktoken
encoding = tiktoken.get_encoding('cl100k_base')
token_splitter = TokenTextSplitter(encoding_name='cl100k_base', chunk_size=55, chunk_overlap=0)
token_docs = show_chunks('token windows', token_splitter.split_text(TEXT))
for i, doc in enumerate(token_docs, 1):
    print(f'Chunk {i}: {len(encoding.encode(doc.page_content))} tokens')
# Alternative: RecursiveCharacterTextSplitter.from_tiktoken_encoder(
#     encoding_name='cl100k_base', chunk_size=55, chunk_overlap=0)
# This combines token-aware sizing with recursive text boundaries.


Token demonstration not executed here: tokenizer download needs network access. Run this cell after installation with internet access.


## 6. Sentence-boundary grouping
Use a punctuation regex as the separator, then merge whole sentences to a character target. This is a lightweight sentence approximation using CharacterTextSplitter, not a linguistic sentence tokenizer.

**Strength:** Readable sentences and no language-model or NLP downloads.

**Weakness:** Abbreviations and punctuation can fool the regex. A single long sentence can exceed the size target.

**Use cases:** Simple English prose demos; use dedicated NLP sentence splitters for harder multilingual text.

**Observe:** no sentence is cut here; several small sentences may share a chunk.


In [8]:
sentences = CharacterTextSplitter(
    separator=r'(?<=[.!?])\s+', is_separator_regex=True,
    keep_separator='end', chunk_size=300, chunk_overlap=0)
show_chunks('sentence grouping', sentences.split_text(TEXT));



sentence grouping: 5 chunks | displayed length unit: characters

CHUNK 1 | 245 characters
Customers can cancel a paid subscription from the billing page at any time. Cancellation stops the next renewal but keeps paid features active until the current billing period ends. Canceling a subscription does not automatically issue a refund.
Whitespace view: 'Customers can cancel a paid subscription from the billing page at any time. Cancellation stops the next renewal but keeps paid features active until the current billing period ends. Canceling a subscription does not automatically issue a refund.'

CHUNK 2 | 239 characters
Refund requests must be submitted to billing support within fourteen days of the original purchase. Include the invoice number so the billing team can locate the payment.

Account security uses a password and an optional authenticator app.
Whitespace view: 'Refund requests must be submitted to billing support within fourteen days of the original purchase. Include the in

## 7. Markdown heading splitting
This fixture adds actual headings, which our running text does not contain. Heading splitters return Documents with section metadata rather than enforcing a size budget.

**Strength:** Keeps section identity and hierarchy in metadata.

**Weakness:** Depends on correct headings; a section may be huge. Stripping headings removes them from chunk text.

**Use cases:** Documentation, handbooks, and Markdown knowledge bases.

**Observe:** metadata identifies the parent and subsection. A second recursive pass bounds long sections and preserves metadata.


The expanded handbook has four main sections and eight subsections. First inspect the complete source and rendered document below; the following cell performs chunking. Header metadata now includes `subsection` so students can see three levels of hierarchy.

In [9]:
from IPython.display import display, Markdown, HTML as HTMLPreview

MARKDOWN = '# Customer Support Handbook\n\nWelcome to the Northstar Workspace support handbook. This fictional guide explains billing, account security, document uploads, and incident reporting. Each section includes policies and practical steps so a support assistant can retrieve a complete answer with its conditions.\n\n## Billing and Subscriptions\n\nCustomers can manage their subscriptions from the billing page. Workspace owners can update a payment method, download invoices, and choose a different plan. Before changing a plan, check which features the workspace currently uses and whether the new plan includes them.\n\n### Canceling a Subscription\n\nCancellation stops the next renewal but does not immediately remove paid features. The workspace remains on its current plan until the paid billing period ends. After that date, the workspace moves to the free plan and the free storage limit applies.\n\nTo cancel, open Settings, select Billing, and choose Cancel subscription. Read the confirmation screen and save the confirmation email. Removing the application from a computer does not cancel the subscription.\n\n### Refund Requests\n\nCanceling a subscription does not automatically issue a refund. Submit a refund request to billing support within fourteen days of the original purchase. Include the invoice number and a short explanation of the issue so the billing team can locate the payment.\n\nDuplicate charges should be reported with the dates and amounts of both transactions. Do not include a complete card number in a support message. The billing team may request additional information through a secure form.\n\n## Account Security\n\nEvery workspace member should use a unique password and enable two-factor authentication. Shared accounts make it difficult to identify who changed a document or approved a payment. Invite each person with their own email address instead.\n\n### Recovering Access\n\nIf an authenticator device is lost, use a previously saved recovery code to sign in. Each recovery code works only once. After recovering access, enroll the replacement device and generate a new set of recovery codes.\n\nIf both the device and recovery codes are unavailable, contact account support for identity verification. Support cannot bypass verification simply because a request is urgent. Never send a password or recovery code in a support ticket.\n\n### Reviewing Workspace Permissions\n\nWorkspace owners should review member permissions every month. Remove former employees and check whether external collaborators still need access. Give billing permissions only to people who manage purchases.\n\n- Viewers can read shared documents.\n- Editors can update documents in their assigned projects.\n- Owners can manage membership, billing, and workspace settings.\n\n## Document Uploads\n\nThe workspace accepts PDF and plain text files up to twenty megabytes per file. Scanned PDFs may need optical character recognition before their content can be searched. A successful upload does not necessarily mean the document is already indexed.\n\n### Troubleshooting Upload Failures\n\nA file above the size limit must be compressed or divided before upload. If a smaller file fails, confirm its format and retry after the network connection is stable. Password-protected PDFs must be unlocked before processing.\n\nA failed upload does not consume the document processing quota. Avoid submitting the same file repeatedly while an earlier attempt is still processing. Check the document status page before starting another attempt.\n\n### Processing and Search Availability\n\nProcessing can take several minutes after an upload succeeds. The status moves from Uploaded to Processing and finally to Searchable. Large documents and complex layouts can take longer than short plain text files.\n\nWhen a document becomes searchable, test a question whose answer appears in the original file. If the answer is missing, inspect the extracted text for empty pages, broken reading order, or missing tables before changing the retrieval configuration.\n\n## Contacting Support\n\nA useful support request includes the workspace identifier, a description of the problem, and the time it occurred. Describe what you expected to happen and what happened instead. Remove private customer information from screenshots before attaching them.\n\n### Reporting a Service Incident\n\nCheck the service status page before reporting a widespread outage. If no incident is listed, include the affected feature, approximate number of users, and any visible error message. Report security concerns through the dedicated security channel rather than a public discussion.\n\n### Following Up on a Ticket\n\nReply to the existing ticket instead of opening multiple requests about the same issue. Add new observations, including whether the problem still occurs on another browser or network. Keep the ticket number for future reference.\n'
print('MARKDOWN FILE: support_handbook.md — complete source BEFORE chunking\n')
print(MARKDOWN)
print('RENDERED MARKDOWN — BEFORE chunking')
display(Markdown(MARKDOWN))

MARKDOWN FILE: support_handbook.md — complete source BEFORE chunking

# Customer Support Handbook

Welcome to the Northstar Workspace support handbook. This fictional guide explains billing, account security, document uploads, and incident reporting. Each section includes policies and practical steps so a support assistant can retrieve a complete answer with its conditions.

## Billing and Subscriptions

Customers can manage their subscriptions from the billing page. Workspace owners can update a payment method, download invoices, and choose a different plan. Before changing a plan, check which features the workspace currently uses and whether the new plan includes them.

### Canceling a Subscription

Cancellation stops the next renewal but does not immediately remove paid features. The workspace remains on its current plan until the paid billing period ends. After that date, the workspace moves to the free plan and the free storage limit applies.

To cancel, open Settings, select Bill

# Customer Support Handbook

Welcome to the Northstar Workspace support handbook. This fictional guide explains billing, account security, document uploads, and incident reporting. Each section includes policies and practical steps so a support assistant can retrieve a complete answer with its conditions.

## Billing and Subscriptions

Customers can manage their subscriptions from the billing page. Workspace owners can update a payment method, download invoices, and choose a different plan. Before changing a plan, check which features the workspace currently uses and whether the new plan includes them.

### Canceling a Subscription

Cancellation stops the next renewal but does not immediately remove paid features. The workspace remains on its current plan until the paid billing period ends. After that date, the workspace moves to the free plan and the free storage limit applies.

To cancel, open Settings, select Billing, and choose Cancel subscription. Read the confirmation screen and save the confirmation email. Removing the application from a computer does not cancel the subscription.

### Refund Requests

Canceling a subscription does not automatically issue a refund. Submit a refund request to billing support within fourteen days of the original purchase. Include the invoice number and a short explanation of the issue so the billing team can locate the payment.

Duplicate charges should be reported with the dates and amounts of both transactions. Do not include a complete card number in a support message. The billing team may request additional information through a secure form.

## Account Security

Every workspace member should use a unique password and enable two-factor authentication. Shared accounts make it difficult to identify who changed a document or approved a payment. Invite each person with their own email address instead.

### Recovering Access

If an authenticator device is lost, use a previously saved recovery code to sign in. Each recovery code works only once. After recovering access, enroll the replacement device and generate a new set of recovery codes.

If both the device and recovery codes are unavailable, contact account support for identity verification. Support cannot bypass verification simply because a request is urgent. Never send a password or recovery code in a support ticket.

### Reviewing Workspace Permissions

Workspace owners should review member permissions every month. Remove former employees and check whether external collaborators still need access. Give billing permissions only to people who manage purchases.

- Viewers can read shared documents.
- Editors can update documents in their assigned projects.
- Owners can manage membership, billing, and workspace settings.

## Document Uploads

The workspace accepts PDF and plain text files up to twenty megabytes per file. Scanned PDFs may need optical character recognition before their content can be searched. A successful upload does not necessarily mean the document is already indexed.

### Troubleshooting Upload Failures

A file above the size limit must be compressed or divided before upload. If a smaller file fails, confirm its format and retry after the network connection is stable. Password-protected PDFs must be unlocked before processing.

A failed upload does not consume the document processing quota. Avoid submitting the same file repeatedly while an earlier attempt is still processing. Check the document status page before starting another attempt.

### Processing and Search Availability

Processing can take several minutes after an upload succeeds. The status moves from Uploaded to Processing and finally to Searchable. Large documents and complex layouts can take longer than short plain text files.

When a document becomes searchable, test a question whose answer appears in the original file. If the answer is missing, inspect the extracted text for empty pages, broken reading order, or missing tables before changing the retrieval configuration.

## Contacting Support

A useful support request includes the workspace identifier, a description of the problem, and the time it occurred. Describe what you expected to happen and what happened instead. Remove private customer information from screenshots before attaching them.

### Reporting a Service Incident

Check the service status page before reporting a widespread outage. If no incident is listed, include the affected feature, approximate number of users, and any visible error message. Report security concerns through the dedicated security channel rather than a public discussion.

### Following Up on a Ticket

Reply to the existing ticket instead of opening multiple requests about the same issue. Add new observations, including whether the problem still occurs on another browser or network. Keep the ticket number for future reference.


In [10]:
heading_splitter = MarkdownHeaderTextSplitter(
    headers_to_split_on=[('#', 'guide'), ('##', 'section'), ('###', 'subsection')],
    strip_headers=True)
sections = heading_splitter.split_text(MARKDOWN)
show_chunks('markdown sections', sections)
show_chunks('markdown sections then recursive', recursive.split_documents(sections));


markdown sections: 13 chunks | displayed length unit: characters

CHUNK 1 | 277 characters
Metadata: {'guide': 'Customer Support Handbook'}
Welcome to the Northstar Workspace support handbook. This fictional guide explains billing, account security, document uploads, and incident reporting. Each section includes policies and practical steps so a support assistant can retrieve a complete answer with its conditions.
Whitespace view: 'Welcome to the Northstar Workspace support handbook. This fictional guide explains billing, account security, document uploads, and incident reporting. Each section includes policies and practical steps so a support assistant can retrieve a complete answer with its conditions.'

CHUNK 2 | 271 characters
Metadata: {'guide': 'Customer Support Handbook', 'section': 'Billing and Subscriptions'}
Customers can manage their subscriptions from the billing page. Workspace owners can update a payment method, download invoices, and choose a different plan. Before chan

## 8. HTML heading splitting
Use a small HTML fixture because document tags and headings are necessary for this method.

**Strength:** Extracts readable content and attaches heading context.

**Weakness:** Not a browser renderer; malformed HTML or layout-heavy pages can produce surprising results. No fixed section size guarantee.

**Use cases:** Web documentation and exported HTML articles.

**Observe:** HTML tags disappear and heading metadata remains. Exact text grouping can vary with library versions.


The expanded HTML help page includes paragraphs, lists, and three heading levels. Inspect both the full HTML source and rendered page before running the next chunking cell. Lists demonstrate that HTML extraction may normalize formatting; it does not preserve the original page layout.

In [11]:
from IPython.display import display, HTML as HTMLPreview

HTML = '<!DOCTYPE html>\n<html lang="en">\n<head>\n  <meta charset="utf-8">\n  <title>Northstar Workspace Help Center</title>\n</head>\n<body>\n  <main>\n    <h1>Customer Support Handbook</h1>\n    <p>Welcome to the Northstar Workspace help center. This fictional article explains billing, account security, document uploads, and support requests. Use the section headings to locate the relevant policy before making a change.</p>\n\n    <h2>Billing and Subscriptions</h2>\n    <p>Workspace owners manage plans, payment methods, and invoices from the billing page. Before changing plans, review the features and storage limits your team currently needs.</p>\n    <h3>Canceling a Subscription</h3>\n    <p>Cancellation stops the next renewal. Paid features remain available until the current billing period ends, after which the free plan limits apply.</p>\n    <ol>\n      <li>Open Settings and select Billing.</li>\n      <li>Choose Cancel subscription and review the effective date.</li>\n      <li>Confirm the cancellation and save the confirmation email.</li>\n    </ol>\n    <p>Uninstalling the application does not cancel a subscription. If another workspace owner manages billing, coordinate with that owner before making changes.</p>\n    <h3>Refund Requests</h3>\n    <p>Cancellation does not automatically issue a refund. Contact billing support within fourteen days of the original purchase and provide the invoice number and reason for the request.</p>\n    <p>For a duplicate charge, include the dates and amounts of both transactions. Never include a complete card number in an ordinary support message.</p>\n\n    <h2>Account Security</h2>\n    <p>Use a unique password and enable two-factor authentication. Give each team member an individual account so changes can be attributed to the correct person.</p>\n    <h3>Recovering Access</h3>\n    <p>If an authenticator device is lost, use a saved recovery code. Each code works only once. Enroll a replacement device and generate fresh codes after signing in.</p>\n    <p>If both the device and recovery codes are unavailable, contact account support for identity verification. Support cannot skip verification because the request is urgent. Never send a password or recovery code in a ticket.</p>\n    <h3>Reviewing Permissions</h3>\n    <p>Review workspace membership every month. Remove former employees and external collaborators who no longer need access. Limit billing permissions to the people who manage purchases.</p>\n    <ul>\n      <li>Viewers read shared documents.</li>\n      <li>Editors update documents in assigned projects.</li>\n      <li>Owners manage membership, billing, and settings.</li>\n    </ul>\n\n    <h2>Document Uploads</h2>\n    <p>Upload PDF or plain text documents up to twenty megabytes per file. Scanned PDFs may need optical character recognition. Upload completion and search availability are separate stages.</p>\n    <h3>Troubleshooting Upload Failures</h3>\n    <p>Compress or divide files that exceed the size limit. For smaller files, verify the format, unlock password-protected PDFs, and retry after the connection is stable.</p>\n    <p>Failed uploads do not consume the document processing quota. Check the status of an earlier attempt before uploading the same file again.</p>\n    <h3>Processing and Search Availability</h3>\n    <p>The document status progresses from Uploaded to Processing and then Searchable. Complex layouts and large files can take longer to process than short plain text documents.</p>\n    <p>Once processing completes, test a question with a known answer from the file. Inspect extracted text for empty pages, missing tables, or incorrect reading order if the answer cannot be found.</p>\n\n    <h2>Contacting Support</h2>\n    <p>Include the workspace identifier, the time of the problem, and a description of expected and actual behavior. Remove private customer information from screenshots before attaching them.</p>\n    <h3>Reporting a Service Incident</h3>\n    <p>Check the service status page first. If no incident is listed, report the affected feature, number of users, and error message. Send security concerns through the dedicated security channel.</p>\n    <h3>Following Up on a Ticket</h3>\n    <p>Reply to the existing ticket rather than opening duplicate requests. Mention whether the problem occurs in another browser or on another network, and keep the ticket number for reference.</p>\n  </main>\n</body>\n</html>\n'
print('HTML FILE: support_handbook.html — complete source BEFORE chunking\n')
print(HTML)
print('RENDERED HTML — BEFORE chunking')
display(HTMLPreview(HTML))

HTML FILE: support_handbook.html — complete source BEFORE chunking

<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <title>Northstar Workspace Help Center</title>
</head>
<body>
  <main>
    <h1>Customer Support Handbook</h1>
    <p>Welcome to the Northstar Workspace help center. This fictional article explains billing, account security, document uploads, and support requests. Use the section headings to locate the relevant policy before making a change.</p>

    <h2>Billing and Subscriptions</h2>
    <p>Workspace owners manage plans, payment methods, and invoices from the billing page. Before changing plans, review the features and storage limits your team currently needs.</p>
    <h3>Canceling a Subscription</h3>
    <p>Cancellation stops the next renewal. Paid features remain available until the current billing period ends, after which the free plan limits apply.</p>
    <ol>
      <li>Open Settings and select Billing.</li>
      <li>Choose Cancel subscription and

Rendered support handbook HTML

In [12]:
html_splitter = HTMLHeaderTextSplitter(
    headers_to_split_on=[('h1', 'guide'), ('h2', 'section'), ('h3', 'subsection')])
show_chunks('html headings', html_splitter.split_text(HTML));


html headings: 26 chunks | displayed length unit: characters

CHUNK 1 | 25 characters
Metadata: {'guide': 'Customer Support Handbook'}
Customer Support Handbook
Whitespace view: 'Customer Support Handbook'

CHUNK 2 | 225 characters
Metadata: {'guide': 'Customer Support Handbook'}
Welcome to the Northstar Workspace help center. This fictional article explains billing, account security, document uploads, and support requests. Use the section headings to locate the relevant policy before making a change.
Whitespace view: 'Welcome to the Northstar Workspace help center. This fictional article explains billing, account security, document uploads, and support requests. Use the section headings to locate the relevant policy before making a change.'

CHUNK 3 | 25 characters
Metadata: {'guide': 'Customer Support Handbook', 'section': 'Billing and Subscriptions'}
Billing and Subscriptions
Whitespace view: 'Billing and Subscriptions'

CHUNK 4 | 172 characters
Metadata: {'guide': 'Customer Suppor

## 9. Python code-aware recursive splitting
A separate Python fixture is needed to expose class and function boundaries. Language-specific separators prioritize those boundaries.

**Strength:** Often keeps useful code units together better than arbitrary windows.

**Weakness:** Separator heuristics are not an AST parser; long functions or literals can still be cut.

**Use cases:** Code search and code assistants.

**Observe:** compare class/function boundaries with the printed chunks. Size constraints may force splits inside a unit.


In [13]:
PYTHON_SOURCE = '''class BillingService:
    def cancel(self, account):
        # Cancellation stops renewal; it does not issue a refund.
        account.renewal_enabled = False
        return account.paid_until

    def refund_allowed(self, days_since_purchase):
        return days_since_purchase <= 14


def validate_upload(size_mb, extension):
    if extension not in {'.pdf', '.txt'}:
        raise ValueError('Unsupported document format')
    if size_mb > 20:
        raise ValueError('Compress or divide this file before uploading')
    return True
'''
print('SOURCE:\n', PYTHON_SOURCE)
code_splitter = RecursiveCharacterTextSplitter.from_language(
    language=Language.PYTHON, chunk_size=240, chunk_overlap=0)
show_chunks('python code', code_splitter.split_text(PYTHON_SOURCE));


SOURCE:
 class BillingService:
    def cancel(self, account):
        # Cancellation stops renewal; it does not issue a refund.
        account.renewal_enabled = False
        return account.paid_until

    def refund_allowed(self, days_since_purchase):
        return days_since_purchase <= 14


def validate_upload(size_mb, extension):
    if extension not in {'.pdf', '.txt'}:
        raise ValueError('Unsupported document format')
    if size_mb > 20:
        raise ValueError('Compress or divide this file before uploading')
    return True


python code: 4 chunks | displayed length unit: characters

CHUNK 1 | 192 characters
class BillingService:
    def cancel(self, account):
        # Cancellation stops renewal; it does not issue a refund.
        account.renewal_enabled = False
        return account.paid_until
Whitespace view: 'class BillingService:\n    def cancel(self, account):\n        # Cancellation stops renewal; it does not issue a refund.\n        account.renewal_enabled = 

## 10. Recursive JSON splitting
JSON needs a structured fixture. This splitter traverses nested objects rather than cutting the serialized string at arbitrary characters.

**Strength:** Preserves valid JSON fragments and parent key paths.

**Weakness:** Large scalar values can exceed the target. List conversion changes arrays into index-keyed objects; fragments are not complete original records.

**Use cases:** Structured API responses and configuration data.

**Observe:** each printed chunk is valid JSON and retains enclosing keys. Size is based on JSON serialization, not tokens.


In [14]:
JSON_DATA = {'support': {
    'billing': {'cancellation': 'Stops the next renewal.', 'refund': 'Request within fourteen days with invoice number.'},
    'security': {'recovery': 'Use a saved single-use recovery code.', 'verification': 'Contact support if all recovery methods are lost.'},
    'uploads': {'allowed': ['PDF', 'plain text'], 'limit_mb': 20, 'failure': 'Compress or divide oversized files.'}
}}
print('SOURCE:\n', json.dumps(JSON_DATA, indent=2))
json_splitter = RecursiveJsonSplitter(max_chunk_size=180, min_chunk_size=80)
json_chunks = json_splitter.split_json(JSON_DATA, convert_lists=True)
show_chunks('recursive json', [json.dumps(c, indent=2) for c in json_chunks]);


SOURCE:
 {
  "support": {
    "billing": {
      "cancellation": "Stops the next renewal.",
      "refund": "Request within fourteen days with invoice number."
    },
    "security": {
      "recovery": "Use a saved single-use recovery code.",
      "verification": "Contact support if all recovery methods are lost."
    },
    "uploads": {
      "allowed": [
        "PDF",
        "plain text"
      ],
      "limit_mb": 20,
      "failure": "Compress or divide oversized files."
    }
  }
}

recursive json: 3 chunks | displayed length unit: characters

CHUNK 1 | 162 characters
{
  "support": {
    "billing": {
      "cancellation": "Stops the next renewal.",
      "refund": "Request within fourteen days with invoice number."
    }
  }
}
Whitespace view: '{\n  "support": {\n    "billing": {\n      "cancellation": "Stops the next renewal.",\n      "refund": "Request within fourteen days with invoice number."\n    }\n  }\n}'

CHUNK 2 | 179 characters
{
  "support": {
    "security": {
    

## 11. Semantic chunking: support retrieval after layout is lost
**Real use case:** an extracted support article has lost paragraph breaks. A refund question should retrieve cancellation/refund rules together, rather than half a billing rule plus an unrelated security instruction. Semantic chunking looks for changes in meaning between nearby sentence windows even without headings or paragraph layout.

**Strength:** may keep coherent topic passages together despite missing formatting.

**Weakness:** requires embeddings, API access, latency and potentially cost; results depend on the embedding model and threshold. It does not guarantee a particular topic boundary or maximum chunk size. Poor sentence detection and gradual topic transitions can hurt it. It is not automatically better than recursive splitting.

**Use cases:** flattened multi-topic articles, transcripts, and prose where meaning changes without dependable structural markers.

### Configure Gemini (default) or OpenAI
Set the key in your environment or enter it securely when prompted. Keys are never printed or saved in the notebook. The small synthetic source is sent to the selected provider. Change `SEMANTIC_PROVIDER` and rerun this section to compare providers. Select an embedding model supported by your account, not a chat model.

The section skips with a clear message if you leave the key blank. All other demonstrations remain usable. No fake vectors are substituted.


In [15]:
import os
from getpass import getpass

SEMANTIC_PROVIDER = 'gemini'  # Switch to 'openai'.
GEMINI_EMBEDDING_MODEL = os.getenv('GEMINI_EMBEDDING_MODEL', 'gemini-embedding-001')
OPENAI_EMBEDDING_MODEL = os.getenv('OPENAI_EMBEDDING_MODEL', 'text-embedding-3-small')

embeddings = None
if SEMANTIC_PROVIDER == 'gemini':
    key = os.getenv('GOOGLE_API_KEY') or os.getenv('GEMINI_API_KEY')
    key = key or getpass('Gemini API key (Enter to skip semantic section): ')
    if key:
        from langchain_google_genai import GoogleGenerativeAIEmbeddings
        embeddings = GoogleGenerativeAIEmbeddings(model=GEMINI_EMBEDDING_MODEL, google_api_key=key)
elif SEMANTIC_PROVIDER == 'openai':
    key = os.getenv('OPENAI_API_KEY') or getpass('OpenAI API key (Enter to skip): ')
    if key:
        from langchain_openai import OpenAIEmbeddings
        embeddings = OpenAIEmbeddings(model=OPENAI_EMBEDDING_MODEL, api_key=key)
else:
    raise ValueError('Choose gemini or openai.')
print('Semantic provider ready.' if embeddings else 'Skipped: no API key supplied.')


Semantic setup not executed during validation; run this cell with your key.


### Print semantic output next to the recursive baseline
`buffer_size=1` embeds each sentence with its immediate neighbors. A boundary is created when adjacent windows have a cosine distance above the chosen percentile threshold. Here 80 is an illustrative setting, not an optimized value; a higher percentile usually allows fewer breaks. The percentile ranks distances **inside this document**; it is not an 80% similarity score.

**Observe:** whether the cuts align with billing → security → uploads. The output is intentionally not hard-coded. Compare full chunk text with the baseline and check whether refund eligibility stays with its deadline. The experimental package API can change.


In [16]:
print('FLATTENED SOURCE:\n', FLAT_TEXT)
show_chunks('flattened recursive baseline', recursive.split_text(FLAT_TEXT))
if embeddings is not None:
    from langchain_experimental.text_splitter import SemanticChunker
    semantic = SemanticChunker(
        embeddings, buffer_size=1,
        breakpoint_threshold_type='percentile', breakpoint_threshold_amount=80)
    semantic_docs = semantic.create_documents([FLAT_TEXT])
    show_chunks(f'semantic {SEMANTIC_PROVIDER} percentile 80', semantic_docs)
else:
    print('Semantic output not generated. Supply a provider key and rerun the configuration cell.')


FLATTENED SOURCE:
 Customers can cancel a paid subscription from the billing page at any time. Cancellation stops the next renewal but keeps paid features active until the current billing period ends. Canceling a subscription does not automatically issue a refund. Refund requests must be submitted to billing support within fourteen days of the original purchase. Include the invoice number so the billing team can locate the payment. Account security uses a password and an optional authenticator app. If an authenticator device is lost, a saved recovery code can restore access. Each recovery code works only once and should be stored somewhere private. If both the device and recovery codes are unavailable, contact account support for identity verification. Never send a password or a recovery code in a support ticket. File uploads accept PDF and plain text documents up to twenty megabytes per file. A file above that limit must be compressed or divided before it can be uploaded. A failed upl

### Check the retrieval motivation without building a retriever
This is a **literal phrase check**, not semantic retrieval or a retrieval-quality score. It makes boundary damage visible by showing chunks containing each clue. Ask students whether each candidate contains enough context to answer the corresponding question.

**Expected:** recursive and semantic chunks may contain different amounts of neighboring context. If semantic was skipped, only the baseline appears.


In [17]:
questions = {
    'Can I get a refund after cancellation?': 'Refund requests',
    'What if I lose my recovery codes too?': 'both the device',
    'Why did my large upload fail?': 'above that limit',
}
for name, docs in results.items():
    if not (name == 'flattened recursive baseline' or name.startswith('semantic ')):
        continue
    print('\nMETHOD:', name)
    for question, clue in questions.items():
        print('\nQUESTION:', question)
        hits = [(i, d.page_content) for i, d in enumerate(docs, 1) if clue in d.page_content]
        for i, text in hits:
            print(f'Candidate chunk {i}: {text}')
        if not hits:
            print('No intact literal clue found; it may cross a boundary.')



METHOD: flattened recursive baseline

QUESTION: Can I get a refund after cancellation?
Candidate chunk 2: automatically issue a refund. Refund requests must be submitted to billing support within fourteen days of the original purchase. Include the invoice number so the billing team can locate the payment. Account security

QUESTION: What if I lose my recovery codes too?
No intact literal clue found; it may cross a boundary.

QUESTION: Why did my large upload fail?
Candidate chunk 5: megabytes per file. A file above that limit must be compressed or divided before it can be uploaded. A failed upload does not consume the document processing quota. If a smaller file still fails, check its format and


## Compare the visible results
Counts and character lengths describe the chunks; they do not measure retrieval quality. Token, character, and structural methods use different units or constraints. The Markdown two-pass example demonstrates combining strategies.


In [18]:
print(f'{"Method":48s} {"Chunks":>6s} {"Min chars":>10s} {"Max chars":>10s}')
for name, docs in results.items():
    sizes = [len(d.page_content) for d in docs]
    print(f'{name:48s} {len(docs):6d} {min(sizes, default=0):10d} {max(sizes, default=0):10d}')


Method                                           Chunks  Min chars  Max chars
fixed characters                                      6        133        220
paragraph separators                                  3        388        425
recursive characters                                  6        176        219
fixed with overlap                                    7        213        220
sentence grouping                                     5        198        289
markdown sections                                    13        228        484
markdown sections then recursive                     32          5        220
html headings                                        26         15        459
python code                                           4         11        233
recursive json                                        3        162        196
flattened recursive baseline                          6        146        217


## How to choose, and classroom experiments
| Need | Starting point | Watch for |
|---|---|---|
| General prose | Recursive character | Sentence cuts and lost context |
| Fixed size baseline | Character windows | Broken words |
| Clean paragraph units | Separator-based character | Oversized paragraphs |
| Model context budget | Token-aware recursive | Correct target tokenizer |
| Sentence integrity | Sentence grouping / NLP splitters | Abbreviations, long sentences |
| Documentation hierarchy | Markdown/HTML headings + recursive pass | Metadata retained during retrieval |
| Code | Language-aware recursive | Heuristics do not parse syntax |
| Structured records | Recursive JSON | Large scalar fields and array conversion |
| Missing layout, changing topics | SemanticChunker | Cost, thresholds, variable chunk size |

1. Increase recursive size from 220 to 400. Which chunks now contain complete answer conditions?
2. Compare overlap 0, 30, and 80. How much text is duplicated?
3. Change semantic percentile 80 to 60 or 95. Each rerun may make new embedding API calls. Are the boundaries more useful, or merely different?
4. Change Gemini to OpenAI. Model-dependent boundaries need not agree.
5. For a real RAG system, compare retrieval recall, answer correctness, latency, and cost on labeled questions. Chunk count alone cannot identify a winner.

### Troubleshooting
- `ModuleNotFoundError`: run the install cell in this kernel, then restart it.
- First `tiktoken` call may download its encoding data; cache it before an offline classroom session.
- Semantic 401/403: check API key and provider permissions. 404: choose an available embedding model. 429: check quota/rate limits and retry later.
- One semantic chunk or unexpected boundaries: inspect sentence punctuation, topic transitions, model, and percentile; this is not necessarily an error.
- Oversized paragraph/sentence/JSON scalar: those methods do not guarantee a hard size cap. Add a bounded recursive second pass when needed.
- No semantic output without a key is expected. Other methods require no API calls after dependencies and tokenizer data are installed.

### Official sources
Checked 2026-11-06. The notebook covers representative families, not every language/tokenizer integration.
- [LangChain text splitters](https://docs.langchain.com/oss/python/integrations/splitters)
- [SemanticChunker implementation and threshold options](https://github.com/langchain-ai/langchain-experimental/blob/main/libs/experimental/langchain_experimental/text_splitter.py)
- [Google Gemini embedding integration](https://docs.langchain.com/oss/python/integrations/embeddings/google_generative_ai)
- [OpenAI embedding integration](https://docs.langchain.com/oss/python/integrations/text_embedding/openai)
- [Recursive JSON implementation](https://github.com/langchain-ai/langchain/blob/master/libs/text-splitters/langchain_text_splitters/json.py)
